# config

> PP-OCRv6 medium, as the PaddleOCR training configs describe it.


In [ ]:
#| default_exp config


#| export
Detection uses PPLCNetV4, a RepLKPAN neck with IntraCL, and the inference DB head. Recognition uses the same backbone with height-only downsampling, then a LightSVTR neck and a CTC head. The NRTR head exists only while training, so it is not in these graphs.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass


## Blocks


A block is `(kernel, in_channels, out_channels, stride, use_se)`. Stride is a pair when the recognizer downsamples height and keeps the width.


In [ ]:
#| export
# kernel, in_channels, out_channels, stride, use_se
Block = tuple[int, int, int, int | tuple[int, int], bool]


In [ ]:
#| export
DET_BLOCKS: dict[str, list[Block]] = {
    "s1": [(3, 128, 128, 1, True), (3, 128, 128, 1, False)],
    "s2": [
        (3, 128, 256, 2, False),
        (3, 256, 256, 1, True),
        (3, 256, 256, 1, False),
    ],
    "s3": [
        (3, 256, 512, 2, False),
        (3, 512, 512, 1, True),
        (3, 512, 512, 1, False),
        (3, 512, 512, 1, True),
        (3, 512, 512, 1, False),
    ],
    "s4": [
        (3, 512, 896, 2, False),
        (3, 896, 896, 1, True),
        (3, 896, 896, 1, False),
    ],
}


Detection stages `s1` to `s4` are the four maps the neck reads. Total stride at the end of the stem plus `s1` is 4, then each later stage halves both sides.


In [ ]:
#| export
REC_BLOCKS: dict[str, list[Block]] = {
    "b2": [(3, 128, 128, 1, True)],
    "b3": [
        (3, 128, 256, 1, False),
        (3, 256, 256, 1, False),
        (3, 256, 256, 1, True),
    ],
    "b4": [
        (3, 256, 512, (2, 1), False),
        (3, 512, 512, 1, True),
        (3, 512, 512, 1, False),
        (3, 512, 512, 1, True),
        (3, 512, 512, 1, False),
        (3, 512, 512, 1, True),
        (3, 512, 512, 1, False),
    ],
    "b5": [
        (3, 512, 768, (2, 1), False),
        (3, 768, 768, 1, True),
        (3, 768, 768, 1, False),
    ],
}


Recognition never strides in width inside the backbone. Height is reduced twice, by the `(2, 1)` blocks, and the rest of the way by the pool in front of the neck.


In [ ]:
from fastcore.test import test_eq

test_eq(DET_BLOCKS["s1"][0], (3, 128, 128, 1, True))
test_eq(REC_BLOCKS["b4"][0][3], (2, 1))
test_eq(len(REC_BLOCKS["b4"]), 7)
REC_BLOCKS["b5"][-1]


(3, 768, 768, 1, False)

## Detector


In [ ]:
#| export
@dataclass(frozen=True)
class DetectorConfig:
    stem_mid: int = 64
    stem_out: int = 128
    # RepLKPAN projects every level to this width, then to width // 4.
    fpn_channels: int = 256
    replk_kernel: int = 9
    head_channels: int = 64
    se_alpha: float = 1.0 / 6.0


RepLKPAN uses a 9×9 reparameterized depthwise kernel. The squeeze-excite gate is a hardsigmoid with α = 1/6.


In [ ]:
from fastcore.test import test_eq

test_eq(DetectorConfig().fpn_channels, 256)
test_eq(DetectorConfig().se_alpha, 1 / 6)
DetectorConfig()


DetectorConfig(stem_mid=64, stem_out=128, fpn_channels=256, replk_kernel=9, head_channels=64, se_alpha=0.16666666666666666)

## Recognizer


In [ ]:
#| export
@dataclass(frozen=True)
class RecognizerConfig:
    stem_mid: int = 64
    stem_out: int = 128
    backbone_channels: int = 768
    svtr_dim: int = 192
    svtr_depth: int = 2
    svtr_heads: int = 8
    mlp_ratio: float = 4.0
    local_kernel: int = 7
    num_classes: int = 18710
    image_height: int = 48
    image_width: int = 320
    se_alpha: float = 1.0 / 6.0
    # Conv batch-norm and the transformer block norms both use 1e-5.
    # The norm after the last block uses 1e-6.
    bn_eps: float = 1e-5
    final_norm_eps: float = 1e-6


The CTC head has 18710 classes: the PP-OCRv6 dictionary, plus the blank. The recognizer's image is 48×320. Two global transformer blocks run at width 192. The norm after the last block is tighter than the norms inside the blocks, 1e-6 rather than 1e-5.


In [ ]:
from fastcore.test import test_eq

rec = RecognizerConfig()
test_eq((rec.image_height, rec.image_width), (48, 320))
test_eq(rec.svtr_dim % rec.svtr_heads, 0)
test_eq(rec.num_classes, 18710)
rec


RecognizerConfig(stem_mid=64, stem_out=128, backbone_channels=768, svtr_dim=192, svtr_depth=2, svtr_heads=8, mlp_ratio=4.0, local_kernel=7, num_classes=18710, image_height=48, image_width=320, se_alpha=0.16666666666666666, bn_eps=1e-05, final_norm_eps=1e-06)

## Device


The reference graphs are float32 on CPU. A compiled cache can be reloaded without rebuilding the graph.


In [ ]:
#| export
#| notest
from max.dtype import DType
from max.graph import DeviceRef

DEVICE = DeviceRef.CPU()
DTYPE = DType.float32
